# Milestone 5 — combined GPU session: predictions backfill + Experiments Q and I

**One GPU trip for four pieces of work.** Bundled deliberately rather than run as
three sessions, because each one separately would pay the same ~6.5 GB of model
downloads and the same environment setup.

| § | Work | Trains? | Guard |
|---|---|---|---|
| **a** | **C/D predictions backfill** — re-score the six in-domain checkpoints so their per-example predictions land (`DECISION_REGISTER.md` M5-2) | No — inference | none (safe to Run All) |
| **b** | **F predictions backfill** — same six checkpoints against the Notri-Fact holdout | No — inference | none (safe to Run All) |
| **c** | **Experiment Q** — punctuation ablation, XLM-R × 3 seeds (M2-3, M5-4) | **Yes** | `CONFIRM_Q` + `--confirm-real-run` |
| **d** | **Experiment I** — length ablation, XLM-R × 4 caps × 3 seeds (M5-5) | **Yes** | `CONFIRM_I` + `--confirm-real-run` |

**Sections a and b must reproduce their committed aggregate numbers exactly.**
They re-run a deterministic forward pass over the same checkpoints and the same
splits, so the metrics are expected to be identical to what is already in the
repo — the *predictions* files are the new artefact. Each is followed by a
verification cell that diffs every numeric leaf against the committed copy and
**stops the notebook** if anything moved. That check is not ceremony: `M5-2`
records a backfill that was nearly committed from the wrong interpreter with every
headline number unchanged.

**Sections c and d produce genuinely new results.** Nothing about them may be
quoted before this notebook has run — the repo currently contains no `Q_*.json`
and no `I_xlm-roberta-base_*.json`, by design.

> Experiment I's **classical half is already done** (B/tfidf_svm, all four caps,
> committed 2026-08-21). It is deterministic and needs no GPU, so it was run
> locally. This notebook runs I's XLM-R half only.

## Before you start

Same two secrets as notebook 05, same settings. See
`research/scripts/MILESTONE_5_GPU_HANDOFF_2.md` for the full pre-flight, the
runtime budget and what to bring back.

### On Kaggle
1. **Settings → Accelerator → GPU T4 x2** (or P100). **A GPU is effectively
   mandatory here**, unlike notebook 05 — sections c and d fine-tune 15 models.
2. **Settings → Internet → On.** Off by default.
3. **Add-ons → Secrets**, attached to this notebook: `HF_TOKEN` (**write** — it
   pushes checkpoints and results) and `KAGGLE_API_TOKEN` (Notri-Fact).
4. Set `HF_STAGING_PREFIX` and `REPO_URL` in the restore-state cell below.


## 0. RESTORE STATE — the one cell to re-run after any kernel restart

**If the kernel restarted, died, or you lost your place: run this cell, then carry
on.** It is the only cell you need to re-run.

Idempotent and order-independent. It re-establishes `REPO_DIR`,
`HF_STAGING_PREFIX`, `os.environ["HF_TOKEN"]`, `os.environ["KAGGLE_API_TOKEN"]`
and `sys.path`, cloning or re-syncing only if needed.

It also sets **`CONFIRM_Q = False` and `CONFIRM_I = False`**. The two training
sections refuse to run while those are false, so a stray **Run All** cannot start
15 fine-tuning runs. Restoring state resets them on purpose — recovering from a
crash must never re-arm expensive work.

Two separate flags, not one, for the same reason notebook 05 refused to reuse
`CONFIRM_TRAIN`: a single global would let confirming Q silently arm I as well,
and the two are independently restartable. If Q finishes and I dies, you re-arm
only I.


In [ ]:
# ---- EDIT THESE, then never touch this cell again ----
HF_STAGING_PREFIX = "your-hf-username"   # e.g. "RehmanAyoub"
REPO_URL = "https://github.com/<your-github-username>/<repo-name>.git"
REPO_BRANCH = "main"
# ------------------------------------------------------

# Guards against an accidental Run All. The two TRAINING sections refuse to run
# unless YOU set these to True by hand. Reset here on purpose: restoring state
# must never re-arm expensive work.
#
# Two flags, deliberately independent (and deliberately not notebook 04's
# CONFIRM_TRAIN, which lives in a different notebook and guards different cells).
# Q and I are separately restartable, so confirming one must never arm the other.
CONFIRM_Q = False   # section c — Experiment Q, 3 XLM-R fine-tunes
CONFIRM_I = False   # section d — Experiment I, 12 XLM-R fine-tunes

# Sections a and b are inference-only re-scoring and are NOT guarded: they cannot
# train anything, and their outputs are expected to reproduce what is already
# committed. They do download ~5.4 GB of checkpoints, which is the only cost of
# running them unintentionally.

# Identifies THIS notebook document, for the stale-notebook guard below.
NOTEBOOK_REVISION = 1

import os
import subprocess
import sys

assert HF_STAGING_PREFIX != "your-hf-username", "Set HF_STAGING_PREFIX first."

ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle")
REPO_DIR = "/kaggle/working/repo" if ON_KAGGLE else "/content/repo"

# Idempotent by construction: `checkout -B` resets onto origin's tip whether the
# clone is fresh, stale or already current. Untracked files survive a branch reset.
os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("cloning...")
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True
    )

os.chdir(REPO_DIR)
subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], check=True)
subprocess.run(["git", "fetch", "--quiet", "origin", REPO_BRANCH], check=True)
subprocess.run(
    ["git", "checkout", "--quiet", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True
)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Credentials, re-read every time rather than "only if unset" — a stale or
# half-set token is exactly the state that fails an hour later at push time.
from research.src.notebook_env import detect_platform, get_secret

os.environ["HF_TOKEN"] = get_secret("HF_TOKEN")
os.environ["HF_STAGING_PREFIX"] = HF_STAGING_PREFIX
os.environ["KAGGLE_API_TOKEN"] = get_secret("KAGGLE_API_TOKEN")

print("state restored")
print(f"  platform      : {detect_platform()}")
print(f"  REPO_DIR      : {REPO_DIR}  (cwd={os.getcwd()})")
subprocess.run(["git", "log", "--oneline", "-1"], check=True)
print(f"  HF_TOKEN      : {'set' if os.environ.get('HF_TOKEN') else 'MISSING'}")
print(f"  KAGGLE_API_TOKEN: {'set' if os.environ.get('KAGGLE_API_TOKEN') else 'MISSING'}")
print(f"  HF_PREFIX     : {os.environ['HF_STAGING_PREFIX']}")
print(f"  CONFIRM_Q     : {CONFIRM_Q}  <- section c refuses to run while False")
print(f"  CONFIRM_I     : {CONFIRM_I}  <- section d refuses to run while False")


## 1. Pre-flight — GPU and internet

**Unlike notebook 05, a GPU matters here.** Sections a and b are forward passes and
would merely be slow on CPU; sections c and d fine-tune **15 XLM-R models** and on
CPU would take days, not hours. So this cell records `HAS_GPU`, and the two
training sections refuse to start without one unless you deliberately override.

Internet is checked hard: Kaggle disables it by default, and without it the clone,
the installs, the checkpoint downloads and every push fail.


In [ ]:
import os, socket, subprocess, sys

ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle")

# --- GPU ---------------------------------------------------------------------
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
HAS_GPU = out.returncode == 0

if not HAS_GPU:
    print(
        "WARNING: no GPU detected.\n"
        "  Sections a and b (inference) will still work and give IDENTICAL\n"
        "  numbers, just slowly.\n"
        "  Sections c and d FINE-TUNE 15 models and will refuse to start without\n"
        "  a GPU -- on CPU they would take days. To attach one: "
        + ("Settings -> Accelerator -> GPU." if ON_KAGGLE
           else "Runtime -> Change runtime type.")
    )
else:
    print(out.stdout)
    # Pin to one GPU. Kaggle's default is "T4 x2", and with two visible the HF
    # Trainer wraps the model in nn.DataParallel and reads
    # per_device_train_batch_size as PER DEVICE -- making the effective batch 32
    # and halving the 401 optimizer steps/epoch the 3-seed design rests on
    # (DECISION_REGISTER.md M4-3). That is a change to training dynamics, not a
    # free speed-up, and it would be invisible in the committed metrics.
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    print("CUDA_VISIBLE_DEVICES=0 (single GPU, matching the Milestone 4 training runs)")

# --- Internet: hard requirement ----------------------------------------------
try:
    socket.create_connection(("huggingface.co", 443), timeout=10).close()
    print("internet: OK")
except OSError as exc:
    sys.exit(
        f"No outbound internet ({exc}).\n"
        + ("On Kaggle this is the DEFAULT. Settings -> Internet -> On, then rerun."
           if ON_KAGGLE else "Check the connection and rerun.")
    )

print("platform:", "Kaggle" if ON_KAGGLE else "Colab / other")
print("HAS_GPU  :", HAS_GPU)


## 2. Sync the repo and install pinned dependencies

Copied **verbatim** from notebook 05 — these cells carry the M4-3 (platform
detection), M4-4 (numpy force-reinstall) and M4-5 (subprocess gate) fixes, and are
duplicated rather than adapted so the three notebooks cannot drift apart.

The only difference is the filename the stale-notebook guard reads.


In [ ]:
import os, subprocess, sys

# Sync to the branch tip on EVERY run, not just when the directory is missing.
#
# The previous version cloned only if the directory was absent. Restarting the
# session keeps the working directory, so on any rerun the clone was skipped and the
# repo silently stayed at whatever commit it was first cloned at — no output said so.
# Two sources of truth (notebook cells vs repo code) then drift apart with nothing
# reporting it. `checkout -B` resets the local branch onto origin's tip, which is
# idempotent and safe here: research/data/raw/ is gitignored, and untracked files
# (downloaded corpora, produced metrics) are left alone by a hard branch reset.
#
# Plain git throughout. Nothing in this cell is platform-specific beyond REPO_DIR,
# so it behaves identically on Colab and Kaggle — but Kaggle needs internet ENABLED
# for the clone and fetch to work at all (checked in the pre-flight cell).
os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True
    )

os.chdir(REPO_DIR)
subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], check=True)
subprocess.run(["git", "fetch", "--quiet", "origin", REPO_BRANCH], check=True)
subprocess.run(
    ["git", "checkout", "--quiet", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True
)

# `import research.src...` needs the repo root on sys.path. Colab's IPython puts the
# CWD there implicitly; Kaggle's starts in /kaggle/working and does not.
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("repo synced to:")
subprocess.run(["git", "log", "--oneline", "-1"], check=True)


In [ ]:
# Stale-notebook guard. Compares the notebook YOU are running against the copy just
# synced from the repo, and stops if yours is older.
#
# Why this exists: on 2026-08-15 a run had a correctly-updated repo but a pre-fix
# notebook tab. The cells issued old commands against new code, so the data check ran
# unscoped and the smoke test hit the torchvision error the new cells prevent. Nothing
# reported the mismatch; it looked like the fixes had simply not worked.
import json, re

with open("research/notebooks/06_milestone5_combined.ipynb", encoding="utf-8") as fh:
    committed = json.load(fh)

marker = re.compile(r"NOTEBOOK" + r"_REVISION\s*=\s*(\d+)")
found = [
    int(m.group(1))
    for cell in committed["cells"]
    for m in marker.finditer("".join(cell["source"]))
]
repo_revision = max(found) if found else 0

if repo_revision > NOTEBOOK_REVISION:
    raise SystemExit(
        f"STALE NOTEBOOK — you are running revision {NOTEBOOK_REVISION}, the repo has "
        f"revision {repo_revision}.\n\n"
        "Restarting the runtime restarts the kernel; it does NOT reload the notebook "
        "source in an already-open tab.\n"
        "Fix: close this tab, re-open the notebook from GitHub "
        "(File -> Open notebook -> GitHub tab -> this repo), and run from the top."
    )

print(f"notebook revision {NOTEBOOK_REVISION}, repo revision {repo_revision} — in sync")


In [ ]:
# Step 1 of 3 — remove the image's preinstalled torchvision/torchaudio BEFORE
# installing. Applies to Kaggle as much as to Colab: Kaggle's notebook image is
# built FROM the Colab runtime image, so it ships the same preinstalled copies.
#
# They are compiled against whatever torch the image shipped with. The install below
# moves torch to this project's pin, and the leftovers then register C++ ops against
# the wrong ABI, so `from transformers import Trainer` dies with
#     RuntimeError: operator torchvision::nms does not exist
# transformers guards that import with is_torchvision_available(), which only checks
# whether the package is INSTALLED, not whether it imports — a present-but-broken
# copy passes the guard and raises RuntimeError, which nothing on that path catches.
# With torchvision absent the guard is simply False and the block is skipped.
# This project does zero vision and zero audio work. See research/requirements.txt.
#
# pip will warn that fastai/timm now have an unsatisfied torchvision requirement.
# That is expected and harmless — nothing in this pipeline imports them.
!pip uninstall -y -q torchvision torchaudio

# Step 2 of 3 — install the pinned set (REPRODUCIBILITY.md Section 1). On Linux
# (both platforms are Linux) the plain torch pin resolves to the CUDA build,
# which is what a T4 or a P100 needs.
!pip install -q -r research/requirements.txt

# Step 3 of 3 — rewrite numpy and scipy COMPLETELY, over the top of the image's
# copies. Same class of problem as the torchvision block above (a preinstalled
# package left in an inconsistent state), but a different mechanism needing a
# different remedy — and it is NOT a scipy/numpy version conflict, despite the
# traceback looking exactly like one. See DECISION_REGISTER.md M4-4.
#
# What actually failed on Kaggle on 2026-08-19, at `from transformers import Trainer`:
#
#     ImportError: cannot import name '_center' from 'numpy._core.umath'
#                  (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)
#
# That import is INSIDE numpy, not at a scipy/numpy boundary. numpy/_core/umath.py
# is a pure-Python shim that re-exports from the COMPILED _multiarray_umath
# extension, and `_center` is a ufunc living in that binary.
#
# CORRECTED by M4-5: the mixture is in MEMORY, not on disk. The image imports numpy
# at kernel boot; pip then correctly replaces it on disk; the kernel keeps the old
# module objects. The lazy numpy.char/_core.strings then load from the NEW files
# against the OLD cached umath. Both pip and the kernel are right at once — which is
# why the 2026-08-19 run reported numpy 2.0.2 while pip reported 2.5.2 installed.
# The environment gate below therefore runs in a subprocess, where no stale modules
# exist. This force-reinstall is KEPT as cheap insurance against a genuinely partial
# on-disk install, which would look identical from inside the kernel.
#
# scipy is only the messenger. Plain `import numpy` does NOT load the affected
# submodules — numpy.char and numpy._core.strings are lazy. `from numpy import *`
# DOES, and scipy's array_api_compat shim is the first thing in the process to run
# it, which is why every earlier cell, and torch itself, imported fine.
#
# scipy is therefore NOT uninstalled the way torchvision is: it is genuinely
# required. `import sklearn.metrics` — the path every metrics file this project
# writes goes through — pulls in 493 scipy modules, and scikit-learn declares scipy
# as a hard dependency. Removing it would break the run outright.
#
# --force-reinstall rewrites every file of both packages instead of skipping them as
# "already satisfied", which is what repairs the mixed install; --no-deps keeps it
# surgical, so nothing else in the resolved set is disturbed.
!pip install -q --force-reinstall --no-deps numpy==2.5.2 scipy==1.18.0


In [ ]:
# Environment gate — runs in a FRESH SUBPROCESS, not in this kernel. That is the
# fix for DECISION_REGISTER.md M4-5, not a stylistic preference.
#
# Kaggle and Colab import numpy at kernel boot. The dependency cell above then
# replaces numpy on disk, but this kernel keeps the module objects it already
# holds. numpy.char and numpy._core.strings are LAZY, so the first
# `from numpy import *` after the install reads those two files from the NEW numpy
# while numpy._core.umath is still the OLD cached one, and the new strings.py asks
# for a `_center` ufunc the old compiled extension does not have:
#
#     ImportError: cannot import name '_center' from 'numpy._core.umath'
#
# pip and the kernel are BOTH right at the same time, which is why the 2026-08-19
# run reported numpy 2.0.2 (the image's, held in memory) while pip correctly
# reported 2.5.2 installed. Reproduced locally; there is no shadow install and no
# corrupt install on disk.
#
# Every real step below already runs as its own `python -m ...` process, so all of
# them read the freshly installed packages and were never affected. Running the
# gate the same way makes it check the environment the TRAINING actually uses,
# instead of this kernel's stale view of it — and removes any need to restart the
# session mid-notebook.
#
# subprocess.run + an explicit raise, rather than a bare `!python`: a `!` command's
# non-zero exit does NOT stop "Run All", so a failed gate would otherwise scroll by
# and the smoke test would run anyway.
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "research/scripts/check_gpu_env.py"],
    cwd=REPO_DIR,
)

if result.returncode != 0:
    raise SystemExit(
        "Environment gate FAILED — see the checks above. Do not continue to the "
        "smoke test; fix the reported items first."
    )


## 3. Data check — **both** corpora

Both are needed, for different sections: Ax-to-Grind because every committed split
index resolves against it (sections a, c-training, d), and Notri-Fact because it is
section b's and section c's evaluation target.

`research/data/raw/` is gitignored, so this re-downloads on every fresh session.


In [ ]:
# Both datasets — no `--only` filter, unlike notebook 04.
!python -m research.src.data.download

# download.py regenerates MANIFEST.sha256 from whatever is on disk, so verifying
# against it straight after a download would be circular. Restore the COMMITTED
# manifest first — that is the actual dataset-version anchor
# (REPRODUCIBILITY.md Section 3).
!git checkout -- research/data/raw/MANIFEST.sha256

# Full integrity + schema checks, both corpora. If the cross-dataset dedup gate
# were ever to change, this is where it would surface before any scoring happens.
!python -m pytest research/tests/test_raw_data_integrity.py -q
!python -m research.src.data.validate


## a. C/D predictions backfill — in-domain, inference only

Re-scores the six Milestone 4 checkpoints (mBERT + XLM-R × seeds {42, 123, 2026})
on the Ax-to-Grind **val and test** splits, writing the per-example predictions
that `REPRODUCIBILITY.md` Section 6 has required since Milestone 1 and that
`DECISION_REGISTER.md` **M5-2** records as missing.

**Not guarded.** It cannot train anything, and its aggregate outputs are expected
to reproduce what is already committed byte for byte. Its only cost if run
unintentionally is ~5.4 GB of checkpoint downloads.

**What is new here is only the `.predictions.jsonl` files.** The `.json` metrics
files are rewritten with identical numbers — `load_best_model_at_end` means the
checkpoint on the Hub *is* the model that produced them, and scoring is `argmax`
over logits with no sampling. The next cell proves that rather than assuming it.

Predictions are pushed to the Hub alongside their metrics as each checkpoint
finishes (M4-6), so a mid-run disconnect costs only the checkpoint in flight.


In [ ]:
# Inference only — no training, no checkpoint writes. Not guarded.
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "research.src.models.evaluate_checkpoint",
     "--experiments", "C", "D"],
    cwd=REPO_DIR,
)
if result.returncode != 0:
    raise SystemExit(
        "C/D backfill failed — see the output above. Any checkpoint that finished "
        "is already on the Hub (M4-6); re-running is safe and simply recomputes."
    )
print("\nC/D backfill complete. Run the verification cell below before continuing.")


### a-verify — the aggregates must not have moved

`M5-2` is the reason this cell exists. That backfill was nearly committed from the
wrong interpreter with **every headline number unchanged**; it was caught by
diffing the full JSON tree rather than trusting the macro-F1. So this walks every
numeric leaf of every regenerated file against the copy committed in git, and
**stops the notebook** on any difference.

`timestamp_utc`, `git_commit` and the `run_metadata.hardware` block are expected to
change and are excluded. Everything else must be identical.


In [ ]:
# Diff every regenerated C/D metrics file against its committed copy.
import json
import subprocess
from pathlib import Path

VOLATILE = {"timestamp_utc", "git_commit", "hardware", "platform",
            "python_version", "sklearn_version", "numpy_version",
            "evaluation_only_recovery", "truncation_evaluated", "checkpoint_selection"}


def numeric_leaves(node, prefix=""):
    """Yield (path, value) for every number in the tree, skipping volatile keys."""
    if isinstance(node, dict):
        for key, value in node.items():
            if key in VOLATILE:
                continue
            yield from numeric_leaves(value, f"{prefix}.{key}")
    elif isinstance(node, list):
        for index, value in enumerate(node):
            yield from numeric_leaves(value, f"{prefix}[{index}]")
    elif isinstance(node, bool):
        yield prefix, node
    elif isinstance(node, (int, float)):
        yield prefix, node


metrics_dir = Path(REPO_DIR) / "research" / "results" / "metrics"
regenerated = sorted(metrics_dir.glob("C_*.json")) + sorted(metrics_dir.glob("D_*.json"))
assert regenerated, "no C_*/D_*.json found — did the backfill cell actually run?"

problems, checked = [], 0
for path in regenerated:
    rel = path.relative_to(REPO_DIR).as_posix()
    committed = subprocess.run(
        ["git", "show", f"HEAD:{rel}"], cwd=REPO_DIR, capture_output=True, text=True
    )
    if committed.returncode != 0:
        problems.append(f"{path.name}: NOT in git at HEAD (unexpected new file)")
        continue

    before = dict(numeric_leaves(json.loads(committed.stdout)))
    after = dict(numeric_leaves(json.loads(path.read_text(encoding="utf-8"))))
    checked += 1

    for key in sorted(set(before) | set(after)):
        old, new = before.get(key), after.get(key)
        if old is None or new is None:
            problems.append(f"{path.name}: {key} appeared/disappeared ({old!r} -> {new!r})")
        elif old != new:
            problems.append(f"{path.name}: {key} {old!r} -> {new!r}")

    sibling = path.with_name(path.name.replace(".json", ".predictions.jsonl"))
    record = json.loads(path.read_text(encoding="utf-8"))
    if record["split"] == "test":
        if not sibling.exists():
            problems.append(f"{path.name}: predictions sibling MISSING — the point of this run")
        else:
            rows = sibling.read_text(encoding="utf-8").strip().splitlines()
            if len(rows) != record["n_examples"]:
                problems.append(
                    f"{sibling.name}: {len(rows)} rows != n_examples {record['n_examples']}"
                )

print(f"checked {checked} file(s)")
if problems:
    for p in problems[:40]:
        print("  !!", p)
    raise SystemExit(
        f"\n{len(problems)} discrepancy(ies) against the committed C/D results.\n"
        "These runs were supposed to REPRODUCE, not change, the aggregates. Do not\n"
        "continue and do not commit — investigate first (M5-2 is the precedent)."
    )
print("all C/D aggregates reproduce the committed values exactly; predictions present")


## b. F predictions backfill — cross-dataset, inference only

The same six checkpoints against the full **Notri-Fact holdout** (13,355 rows).
Identical reasoning to section a: F's aggregate numbers are already committed and
must reproduce exactly; the `.predictions.jsonl` files are the new artefact, and
they are what unblocks `DECISION_REGISTER.md` **M5-3**'s population-level error
sample and Experiment O.

Not guarded. The checkpoints are already cached locally by section a, so this adds
no download.


In [ ]:
# Inference only, on checkpoints section a already downloaded. Not guarded.
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "research.src.experiments.run_cross_dataset",
     "--models", "transformer", "--directions", "F"],
    cwd=REPO_DIR,
)
if result.returncode != 0:
    raise SystemExit(
        "F backfill failed — see the output above. Any seed that finished is "
        "already on the Hub (M4-6); re-running is safe."
    )
print("\nF backfill complete. Run the verification cell below before continuing.")


In [ ]:
# Same full-tree diff as a-verify, against the committed F results.
#
# The helper is redefined rather than reused from the a-verify cell on purpose:
# these sections are independently re-runnable, and a NameError from running
# section b alone after a kernel restart is exactly the kind of mid-session
# annoyance this project keeps logging.
import json
import subprocess
from pathlib import Path

VOLATILE = {"timestamp_utc", "git_commit", "hardware", "platform",
            "python_version", "sklearn_version", "numpy_version",
            "evaluation_only_recovery", "truncation_evaluated", "checkpoint_selection"}


def numeric_leaves(node, prefix=""):
    if isinstance(node, dict):
        for key, value in node.items():
            if key in VOLATILE:
                continue
            yield from numeric_leaves(value, f"{prefix}.{key}")
    elif isinstance(node, list):
        for index, value in enumerate(node):
            yield from numeric_leaves(value, f"{prefix}[{index}]")
    elif isinstance(node, bool):
        yield prefix, node
    elif isinstance(node, (int, float)):
        yield prefix, node


metrics_dir = Path(REPO_DIR) / "research" / "results" / "metrics"
# Transformer files only — F's classical half was computed locally and is untouched.
regenerated = [p for p in sorted(metrics_dir.glob("F_*.json")) if "seed" in p.name]
assert regenerated, "no F_*seed*.json found — did the F backfill cell actually run?"

problems, checked = [], 0
for path in regenerated:
    rel = path.relative_to(REPO_DIR).as_posix()
    committed = subprocess.run(
        ["git", "show", f"HEAD:{rel}"], cwd=REPO_DIR, capture_output=True, text=True
    )
    if committed.returncode != 0:
        problems.append(f"{path.name}: NOT in git at HEAD (unexpected new file)")
        continue

    before = dict(numeric_leaves(json.loads(committed.stdout)))
    after = dict(numeric_leaves(json.loads(path.read_text(encoding="utf-8"))))
    checked += 1
    for key in sorted(set(before) | set(after)):
        old, new = before.get(key), after.get(key)
        if old is None or new is None:
            problems.append(f"{path.name}: {key} appeared/disappeared ({old!r} -> {new!r})")
        elif old != new:
            problems.append(f"{path.name}: {key} {old!r} -> {new!r}")

    sibling = path.with_name(path.name.replace(".json", ".predictions.jsonl"))
    record = json.loads(path.read_text(encoding="utf-8"))
    if not sibling.exists():
        problems.append(f"{path.name}: predictions sibling MISSING — the point of this run")
    else:
        rows = sibling.read_text(encoding="utf-8").strip().splitlines()
        if len(rows) != record["n_examples"]:
            problems.append(
                f"{sibling.name}: {len(rows)} rows != n_examples {record['n_examples']}"
            )

print(f"checked {checked} file(s)")
if problems:
    for p in problems[:40]:
        print("  !!", p)
    raise SystemExit(
        f"\n{len(problems)} discrepancy(ies) against the committed F results.\n"
        "Do not continue and do not commit — investigate first."
    )
print("all F aggregates reproduce the committed values exactly; predictions present")


## c. Experiment Q — punctuation ablation (GUARDED, TRAINS)

**This cell fine-tunes XLM-R three times.** `DECISION_REGISTER.md` M2-3 and M5-4:
retrain on a punctuation-stripped copy of Ax-to-Grind's training split and re-run
Experiment F's zero-shot evaluation on Notri-Fact, so the macro-F1 delta isolates
the surface-form confound from the domain and length confounds already logged.

Writes `Q_xlm-roberta-base_notri_fact_holdout_seed{42,123,2026}.json` plus
predictions siblings, and pushes checkpoints to **`seed-<n>-punct-ablation`
branches inside the existing XLM-R staging repo** — no new repo.

### Two independent guards, on purpose

`CONFIRM_Q` gates the cell, **and** the runner itself requires `--confirm-real-run`
(`DECISION_REGISTER.md` M5-6 — a unit test once started a real training run
because nothing structural stopped it). The notebook flag alone is not enough:
inside a notebook that can be Run-All'd, a single flag is one accident away from
being ignored. The CLI flag alone is not enough either — it would be satisfied by
this cell every time. Both must line up.

To run: put `CONFIRM_Q = True` in a cell of your own, then run this one.


In [ ]:
# EXPENSIVE — guarded. 3 XLM-R fine-tunes + 3 zero-shot passes over 13,355 rows.
import subprocess
import sys

if not globals().get("CONFIRM_Q", False):
    print(
        "SKIPPED — CONFIRM_Q is not set.\n"
        "This cell FINE-TUNES XLM-R three times (Experiment Q).\n"
        "To run it: put `CONFIRM_Q = True` in a cell above, then re-run this.\n"
        "Nothing has been changed."
    )
elif not globals().get("HAS_GPU", False) and not globals().get("ALLOW_CPU_TRAINING", False):
    print(
        "REFUSED — CONFIRM_Q is set but no GPU was detected.\n"
        "Three XLM-R fine-tunes on CPU would take days, and Kaggle's session cap\n"
        "is 12 hours, so the run would die part-way and waste the quota.\n"
        "Attach a GPU (Settings -> Accelerator), or set ALLOW_CPU_TRAINING = True\n"
        "if you genuinely mean it."
    )
else:
    result = subprocess.run(
        [sys.executable, "-m", "research.src.experiments.punctuation_ablation",
         "--confirm-real-run"],
        cwd=REPO_DIR,
    )
    if result.returncode != 0:
        raise SystemExit(
            "Experiment Q failed — see the output above. Each seed pushes its\n"
            "checkpoint AND its metrics as it finishes (M4-6), so any completed\n"
            "seed is safe; re-running recomputes only what is missing."
        )
    print("\nExperiment Q complete.")


## d. Experiment I — length ablation, XLM-R half (GUARDED, TRAINS)

**This cell fine-tunes XLM-R twelve times** — 4 word caps × 3 seeds.
`DECISION_REGISTER.md` M5-5: retrain on word-capped Ax-to-Grind and re-evaluate
in-domain, sweeping `{25, 50, 100, 200}` with **50 the required replication point**
(Haroon's design, the one comparable to the cited 0.0067 drop). The other three
caps draw Part 31 figure 10's curve.

**I's classical half is already committed** and is not re-run here — B/tfidf_svm is
deterministic and was computed locally: macro-F1 0.8551 / 0.8675 / 0.8726 / 0.8770
at caps 25/50/100/200 against uncapped B's 0.8835.

Writes `I_xlm-roberta-base_ax_to_grind_test_seed{n}_cap{N}.json` plus predictions,
and pushes checkpoints to **`seed-<n>-length-cap-<N>` branches** in the existing
XLM-R staging repo.

**Twelve runs, but not twelve full-length runs.** Capping shortens the sequences,
and attention cost is superlinear in length. Measured `E[batch max]` over the real
training split at batch 16: **405 subword tokens uncapped**, against **41 / 73 /
131 / 236** at caps 25 / 50 / 100 / 200 — so the four caps together cost about
**1.19× one uncapped run**, and all twelve runs about **3.6 uncapped-run
equivalents**. See the handoff doc for the resulting time budget.

Same double guard as section c: `CONFIRM_I` here, `--confirm-real-run` in the
runner.


In [ ]:
# EXPENSIVE — guarded. 12 XLM-R fine-tunes (4 caps x 3 seeds), shortened sequences.
import subprocess
import sys

if not globals().get("CONFIRM_I", False):
    print(
        "SKIPPED — CONFIRM_I is not set.\n"
        "This cell FINE-TUNES XLM-R twelve times (Experiment I, 4 caps x 3 seeds).\n"
        "To run it: put `CONFIRM_I = True` in a cell above, then re-run this.\n"
        "Nothing has been changed."
    )
elif not globals().get("HAS_GPU", False) and not globals().get("ALLOW_CPU_TRAINING", False):
    print(
        "REFUSED — CONFIRM_I is set but no GPU was detected.\n"
        "Twelve XLM-R fine-tunes on CPU are not feasible inside a Kaggle session.\n"
        "Attach a GPU, or set ALLOW_CPU_TRAINING = True if you genuinely mean it."
    )
else:
    # --models transformer: the classical half is already committed (run locally,
    # deterministic, no GPU). Re-running it here would overwrite four committed
    # results with identical numbers for no reason.
    result = subprocess.run(
        [sys.executable, "-m", "research.src.experiments.length_ablation",
         "--models", "transformer", "--confirm-real-run"],
        cwd=REPO_DIR,
    )
    if result.returncode != 0:
        raise SystemExit(
            "Experiment I failed — see the output above. Each (cap, seed) pushes\n"
            "its checkpoint AND metrics as it finishes (M4-6), so completed runs\n"
            "are safe. Re-running recomputes everything; to resume a partial run,\n"
            "pass --caps with only the caps still missing."
        )
    print("\nExperiment I (XLM-R half) complete.")


## e. Disk hygiene — prune local checkpoint directories

**Run this if you are tight on disk, or between sections c and d.** Every training
run keeps its best checkpoint under `checkpoints/<run>/` in the session's working
directory. Fifteen XLM-R runs at roughly 1.1 GB each is **~16 GB**, against
Kaggle's ~20 GB working quota — enough to run out part-way through section d.

Safe: each checkpoint has already been pushed to its own branch on the Hub by the
time its run finishes, and the Hub copy is the canonical artefact
(`REPRODUCIBILITY.md` Section 6). This only removes the local duplicate.

It deliberately does **not** run automatically — deleting model files is not
something a Run All should do silently.


In [ ]:
# Prune local checkpoint dirs whose runs have finished and pushed.
import shutil
from pathlib import Path

PRUNE = False   # set True and re-run to actually delete

checkpoints = Path(REPO_DIR) / "checkpoints"
dirs = sorted(p for p in checkpoints.glob("*") if p.is_dir()) if checkpoints.exists() else []

total = sum(f.stat().st_size for p in dirs for f in p.rglob("*") if f.is_file())
print(f"{len(dirs)} local checkpoint dir(s), {total / 1024**3:.2f} GB")
for p in dirs:
    size = sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
    print(f"  {p.name:<48} {size / 1024**3:>6.2f} GB")

if not PRUNE:
    print("\nPRUNE is False — nothing deleted. Set PRUNE = True above and re-run.")
else:
    for p in dirs:
        shutil.rmtree(p, ignore_errors=True)
    print(f"\ndeleted {len(dirs)} directory(ies). The Hub copies are unaffected.")


## f. Summary — copy these tables back

Three tables, one per new result. The `dominant %` column matters as much as
macro-F1 throughout: a model that learned a dataset-specific shortcut collapses
toward one class on an unseen corpus, which macro-F1 alone can understate.


In [ ]:
import glob
import json
import statistics
from collections import defaultdict
from pathlib import Path

records = [json.load(open(p, encoding="utf-8")) for p in glob.glob("research/results/metrics/*.json")]
by_id = defaultdict(list)
for r in records:
    by_id[r["experiment_id"]].append(r)

in_domain_d = {
    (r["model"], r["seed"]): r["metrics"]["macro_f1"]
    for r in by_id.get("D", []) if r["split"] == "test"
}

# ---- Q: punctuation ablation vs F -------------------------------------------
print("=" * 78)
print("Q — punctuation ablation (Ax-to-Grind -> Notri-Fact, zero-shot)")
print("=" * 78)
q = sorted(by_id.get("Q", []), key=lambda r: r["seed"])
if not q:
    print("  (not run)")
else:
    print(f"{'seed':>6}{'Q macro-F1':>12}{'F macro-F1':>12}{'delta':>9}{'dominant %':>12}{'collapsed':>11}")
    for r in q:
        c = r["run_metadata"]["compares_against"]
        base = c["cross_dataset_baseline"]["macro_f1"]
        print(f"{r['seed']:>6}{r['metrics']['macro_f1']:>12.4f}"
              f"{(base if base is not None else float('nan')):>12.4f}"
              f"{c['delta_macro_f1_q_minus_f']:>9.4f}"
              f"{r['prediction_collapse']['dominant_class_share']:>11.2%}"
              f"{str(r['prediction_collapse']['is_collapsed']):>11}")
    deltas = [r["run_metadata"]["compares_against"]["delta_macro_f1_q_minus_f"] for r in q]
    if len(deltas) > 1:
        print(f"\n  delta mean={statistics.mean(deltas):+.4f} sd={statistics.stdev(deltas):.4f}")
    print("\n  Reading: a delta near zero means the punctuation surface-form mismatch")
    print("  was NOT what drove F's collapse. A large positive delta means it was.")

# ---- I: length ablation curve ------------------------------------------------
print()
print("=" * 78)
print("I — length ablation (Ax-to-Grind in-domain). Figure 10's curve.")
print("=" * 78)
i_records = by_id.get("I", [])
if not i_records:
    print("  (not run)")
else:
    xlmr = defaultdict(list)
    classical = {}
    for r in i_records:
        cap = r["run_metadata"]["ablation"]["cap_words"]
        if r["model"] == "xlm-roberta-base":
            xlmr[cap].append(r)
        else:
            classical[cap] = r

    print(f"{'cap':>6}{'XLM-R mean':>12}{'sd':>8}{'delta vs D':>12}"
          f"{'B (svm)':>10}{'delta vs B':>12}{'primary':>9}")
    for cap in sorted(set(xlmr) | set(classical)):
        runs = xlmr.get(cap, [])
        vals = [r["metrics"]["macro_f1"] for r in runs]
        deltas = [r["run_metadata"]["compares_against"]["delta_macro_f1_i_minus_uncapped"]
                  for r in runs]
        mean = statistics.mean(vals) if vals else float("nan")
        sd = statistics.stdev(vals) if len(vals) > 1 else 0.0
        dmean = statistics.mean(deltas) if deltas else float("nan")
        b = classical.get(cap)
        b_f1 = b["metrics"]["macro_f1"] if b else float("nan")
        b_d = (b["run_metadata"]["compares_against"]["delta_macro_f1_i_minus_uncapped"]
               if b else float("nan"))
        primary = "YES" if cap == 50 else ""
        print(f"{cap:>6}{mean:>12.4f}{sd:>8.4f}{dmean:>12.4f}"
              f"{b_f1:>10.4f}{b_d:>12.4f}{primary:>9}")

    print(f"\n  uncapped D (XLM-R) test macro-F1 by seed: "
          f"{ {s: round(v, 4) for (m, s), v in sorted(in_domain_d.items())} }")
    print("  uncapped B (tfidf_svm) test macro-F1: 0.8835")
    print("\n  Reading: cap 50 is the REPLICATION point. Haroon (2026) reports a")
    print("  0.0067 macro-F1 drop there for XLM-R — that is a LITERATURE value, not")
    print("  ours. Compare it against this run's own cap-50 delta above.")

# ---- backfill completeness ---------------------------------------------------
print()
print("=" * 78)
print("M5-2 backfill completeness — every test/holdout file needs a predictions sibling")
print("=" * 78)
missing = []
for path in sorted(glob.glob("research/results/metrics/*.json")):
    rec = json.load(open(path, encoding="utf-8"))
    if rec["split"] in ("test", "holdout"):
        sib = Path(path).with_name(Path(path).name.replace(".json", ".predictions.jsonl"))
        if not sib.exists():
            missing.append(sib.name)
print(f"  {len(missing)} file(s) still missing predictions"
      + (":" if missing else " — M5-2 is CLOSED"))
for name in missing:
    print("   !!", name)


## g. Package — **the run is not done until these are on the Hub**

`DECISION_REGISTER.md` M4-6. Everything was already pushed as it completed; this is
the sweep that catches anything whose retries failed, verifies against what the Hub
actually **lists**, and stops the notebook if it cannot confirm. Only then does it
build a zip for the Output tab — a convenience copy, never the copy.

Note this pushes the `.predictions.jsonl` files too, not only the `.json` metrics.
For sections a and b those predictions are the *entire* point of the run; leaving
them behind would mean repeating the GPU trip to get them.


In [ ]:
import sys
import zipfile
from pathlib import Path

sys.path.insert(0, REPO_DIR)
from research.src.evaluation.results_push import (  # noqa: E402
    push_result_files,
    verify_results_uploaded,
)
from research.src.notebook_env import deliver_file, working_root  # noqa: E402

metrics_dir = Path(REPO_DIR) / "research" / "results" / "metrics"

# Everything this session could have produced or refreshed, metrics AND predictions.
patterns = ["C_*", "D_*", "F_*", "Q_*", "I_*"]
artefacts = sorted(
    {p for pattern in patterns
     for p in list(metrics_dir.glob(f"{pattern}.json"))
     + list(metrics_dir.glob(f"{pattern}.predictions.jsonl"))}
)
assert artefacts, "nothing to package — did any section actually run?"

print(f"pushing {len(artefacts)} file(s) to the Hub...")
print(f"  {push_result_files(artefacts, subdir='milestone5/metrics')}")

verification = verify_results_uploaded(
    [p.name for p in artefacts], subdir="milestone5/metrics"
)
print(f"  verification: {verification}")

if not verification["verified"]:
    raise SystemExit(
        "RESULTS ARE NOT SAFE YET — "
        f"{len(verification.get('missing', []))} file(s) not on the Hub: "
        f"{verification.get('missing')}\n"
        f"reason: {verification.get('reason', 'see above')}\n\n"
        "Do NOT close this session. Re-run this cell. The results exist in\n"
        f"{metrics_dir}, but only there — the exact state M4-6 exists to prevent."
    )

print(f"\nAll {len(artefacts)} files are on the Hub: {verification['repo_id']}")

archive = working_root() / "milestone5_combined_results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in artefacts:
        zf.write(path, arcname=path.name)
print(f"zipped {len(artefacts)} files -> {archive}")
print(deliver_file(archive))
